In [2]:

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report
)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
DATA_PATH = 'marketing_campaign.csv'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_style('whitegrid')


# ===========================================================================
# 1. Load
# ===========================================================================
def load_data(path):
    df = pd.read_csv(path, sep='\t')
    print(f"Loaded: {df.shape}")
    print(f"Missing: {df.isnull().sum()[df.isnull().sum() > 0].to_dict()}")
    print(f"Response distribution: {df['Response'].value_counts().to_dict()}")
    print(f"Positive rate: {df['Response'].mean():.4f}")
    return df


# ===========================================================================
# 2. Preprocess + Feature Engineering
# ===========================================================================
def preprocess(df):
    """
    Clean data and add derived features.
    Imputation/encoding/scaling are deferred to per-fold pipelines.
    """
    df = df.copy()

    # Parse date and compute tenure
    df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
    ref_date = df['Dt_Customer'].max()
    df['Customer_Tenure_Days'] = (ref_date - df['Dt_Customer']).dt.days

    # Derived features
    df['Age'] = 2014 - df['Year_Birth']
    df['TotalChildren'] = df['Kidhome'] + df['Teenhome']
    df['TotalSpending'] = df[[
        'MntWines', 'MntFruits', 'MntMeatProducts',
        'MntFishProducts', 'MntSweetProducts', 'MntGoldProds'
    ]].sum(axis=1)
    df['TotalPurchases'] = df[[
        'NumDealsPurchases', 'NumWebPurchases',
        'NumCatalogPurchases', 'NumStorePurchases'
    ]].sum(axis=1)
    df['CampaignAcceptedTotal'] = df[[
        'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
        'AcceptedCmp4', 'AcceptedCmp5'
    ]].sum(axis=1)

    # Outliers (business rules)
    n0 = len(df)
    df = df[df['Year_Birth'] >= 1940]               # remove impossibly old
    df = df[(df['Income'].isnull()) | (df['Income'] < 200_000)]
    print(f"Outlier removal: {n0} -> {len(df)} rows")

    # Group rare Marital_Status categories
    df['Marital_Status'] = df['Marital_Status'].replace(
        ['Alone', 'Absurd', 'YOLO'], 'Other')

    # Drop unused columns
    df = df.drop(columns=['Z_CostContact', 'Z_Revenue',
                          'Dt_Customer', 'Year_Birth'])
    return df


# ===========================================================================
# 3. Build per-fold pipeline
# ===========================================================================
def build_pipeline(model, numeric_cols, categorical_cols):
    """
    Pipeline runs INSIDE each CV fold so that:
      - median imputation
      - one-hot encoding
      - StandardScaler
    are all fitted only on training data of that fold (no leakage).
    """
    numeric_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])
    return Pipeline([('prep', preprocessor), ('model', model)])


# ===========================================================================
# 4. Run classification
# ===========================================================================
def run_classification(df):
    y = df['Response']
    X = df.drop(columns=['ID', 'Response'])

    categorical_cols = ['Education', 'Marital_Status']
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    print(f"Features: {len(numeric_cols)} numeric + "
          f"{len(categorical_cols)} categorical")

    # Models: class_weight='balanced' addresses 14.9% positive imbalance
    models = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, class_weight='balanced',
            random_state=RANDOM_STATE),
        'KNN': KNeighborsClassifier(n_neighbors=15),
        'Decision Tree': DecisionTreeClassifier(
            max_depth=6, class_weight='balanced',
            random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    results = []
    for name, model in models.items():
        pipe = build_pipeline(model, numeric_cols, categorical_cols)
        cv_res = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        row = {'Model': name}
        for metric in scoring:
            row[metric] = cv_res[f'test_{metric}'].mean()
        results.append(row)

    results_df = pd.DataFrame(results).set_index('Model')
    print("\n5-Fold CV results:")
    print(results_df.round(4).to_string())

    # Select best by ROC-AUC (robust to imbalance)
    best_name = results_df['roc_auc'].idxmax()
    print(f"\nBest model by ROC-AUC: {best_name}")

    # Hold-out evaluation
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    best_pipe = build_pipeline(
        models[best_name], numeric_cols, categorical_cols)
    best_pipe.fit(X_tr, y_tr)
    y_pred = best_pipe.predict(X_te)
    y_proba = best_pipe.predict_proba(X_te)[:, 1]

    print(f"\nHold-out (20%) report:")
    print(classification_report(y_te, y_pred, digits=4))
    print(f"ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}")
    cm = confusion_matrix(y_te, y_pred)
    print(f"Confusion matrix:\n{cm}")

    # --- Plots ---
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].plot(
        kind='bar', ax=axes[0])
    axes[0].set_title('Classification - 5-Fold CV Comparison')
    axes[0].set_ylabel('Score')
    axes[0].set_ylim(0, 1)
    axes[0].legend(loc='lower right', fontsize=8)
    axes[0].tick_params(axis='x', rotation=20)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes[1].set_title(f'Confusion Matrix - {best_name} (hold-out)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/classification_results.png',
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {OUTPUT_DIR}/classification_results.png")

    results_df.to_csv(f'{OUTPUT_DIR}/classification_cv_scores.csv')
    print(f"Saved: {OUTPUT_DIR}/classification_cv_scores.csv")


# ===========================================================================
# Main
# ===========================================================================
def main():
    print("=" * 60)
    print("Customer Personality Analysis - Classification")
    print("=" * 60)
    df = load_data(DATA_PATH)
    df = preprocess(df)
    run_classification(df)
    print("\nDone.")


if __name__ == '__main__':
    main()


Customer Personality Analysis - Classification
Loaded: (2240, 29)
Missing: {'Income': 24}
Response distribution: {0: 1906, 1: 334}
Positive rate: 0.1491
Outlier removal: 2240 -> 2236 rows
Features: 27 numeric + 2 categorical

5-Fold CV results:
                     accuracy  precision  recall      f1  roc_auc
Model                                                            
Logistic Regression    0.8193     0.4426  0.7844  0.5651   0.8969
KNN                    0.8712     0.6654  0.2814  0.3948   0.8309
Decision Tree          0.7652     0.3566  0.6976  0.4706   0.7503
Random Forest          0.8824     0.6993  0.3772  0.4891   0.8909

Best model by ROC-AUC: Logistic Regression

Hold-out (20%) report:
              precision    recall  f1-score   support

           0     0.9518    0.8294    0.8864       381
           1     0.4397    0.7612    0.5574        67

    accuracy                         0.8192       448
   macro avg     0.6957    0.7953    0.7219       448
weighted avg     0.